# Notebook 07 — Post-Model Figures & Statistics

All four models' Table 3/4/5 numbers **together, in one place** — GB/Linear/Ridge are
loaded from their saved `.joblib`s and LASSO is refit inline (never persisted, same as
before — see notebook `06`), then every model's predictions land in one evaluation
frame.

Covers Table 3 (regression metrics), Table 4 (top-k overlap), Table 5 (GB permutation
importance + LASSO retained/eliminated status), and Section 5.3 (DTDI prediction error,
Table \ref{tab:reliability}). Does **not** carry forward the old notebook's DTDI 4-way
classification analysis (confusion matrices, precision/recall, per-group percentile
thresholds) — that classification scheme was tested and found structurally unreliable at
recovering the upper categories, so dead exploration doesn't belong in the restructured
notebooks either.

## 1. Setup — load models, build the evaluation frame

In [1]:
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance

from src.config import FIGURES_DIR, MODELS_DIR
from src.evaluate import (
    MIN_BASELINE_FOR_DTDI,
    build_evaluation_frame,
    compute_dtdi,
    regression_metrics,
    top_k_zone_hour_overlap,
)
from src.features import (
    apply_standard_scaler,
    build_feature_matrix,
    fit_standard_scaler,
    split_train_test,
    split_train_validation,
)
from src.geospatial import load_processed_zone_hour, load_taxi_zones
from src.modeling.train import (
    LASSO_PARAMS,
    MODEL_REGISTRY,
    fit_lasso_regression,
    load_model,
    tune_lasso_alpha,
)

processed_df = load_processed_zone_hour()
train_df, test_df = split_train_test(processed_df)

models = {name: load_model(MODELS_DIR / f"{name}.joblib") for name in MODEL_REGISTRY}
eval_df = build_evaluation_frame(test_df, models)
print(f"Evaluation rows: {len(eval_df):,}")
print(f"Models loaded: {list(models)}")

2026-08-31 03:25:35.814 | INFO     | src.config:<module>:26 - PROJ_ROOT path is: /Users/debaoli/Desktop/nyc-taxi-project/nyc_taxi_demand_prediction_examiner_final


Evaluation rows: 1,888,340
Models loaded: ['linear_regression', 'ridge', 'gradient_boosting']


## 2. LASSO — tune alpha, refit

Tuned on a held-out validation slice carved from the *training* period only (never the
2026 test set), on the standardized `linear` feature set (LASSO is scale-sensitive).

In [2]:
X_train, y_train = build_feature_matrix(train_df, "linear")
X_test, y_test = build_feature_matrix(test_df, "linear")

train_sub_df, validation_df = split_train_validation(train_df)
X_train_sub, y_train_sub = build_feature_matrix(train_sub_df, "linear")
X_validation, y_validation = build_feature_matrix(validation_df, "linear")

scaler = fit_standard_scaler(X_train_sub)
X_train_sub_scaled = apply_standard_scaler(X_train_sub, scaler)
X_validation_scaled = apply_standard_scaler(X_validation, scaler)

best_alpha, tuning_results = tune_lasso_alpha(
    X_train_sub_scaled, y_train_sub, X_validation_scaled, y_validation
)
print(f"Tuned alpha: {best_alpha:g}")
tuning_results.sort_values("validation_MAE").head(5)

Tuned alpha: 0.001


,alpha,validation_MAE,n_nonzero_coefs
8,0.001000,2.920615,19
7,0.000562,2.921748,25
9,0.001778,2.922472,13
6,0.000316,2.923294,30
5,0.000178,2.924357,35


In [3]:
full_scaler = fit_standard_scaler(X_train)
X_train_scaled = apply_standard_scaler(X_train, full_scaler)
X_test_scaled = apply_standard_scaler(X_test, full_scaler)

lasso_model = fit_lasso_regression(X_train_scaled, y_train, alpha=best_alpha, **LASSO_PARAMS)

lasso_pred = np.clip(np.expm1(lasso_model.predict(X_test_scaled)), a_min=0, a_max=None)
eval_df["pred_lasso"] = pd.Series(lasso_pred, index=X_test_scaled.index)
print(f"LASSO refit at alpha={best_alpha:g}; predictions added to the evaluation frame.")

LASSO refit at alpha=0.001; predictions added to the evaluation frame.


## 3. Table 3 — regression metrics (all four models)

MAE, RMSE, and R2 on the 2026 test set, for all four fitted models.

In [4]:
pred_cols = {
    "gradient_boosting": "pred_gradient_boosting",
    "linear_regression": "pred_linear_regression",
    "ridge": "pred_ridge",
    "lasso": "pred_lasso",
}

table3 = pd.DataFrame(
    {
        name: regression_metrics(eval_df["actual"].to_numpy(), eval_df[col].to_numpy())
        for name, col in pred_cols.items()
    }
).T[["MAE", "RMSE", "R2"]]
print("Table 3 -- regression metrics on the 2026 test set")
table3.round(3)

Table 3 -- regression metrics on the 2026 test set


,MAE,RMSE,R2
gradient_boosting,2.200,10.087,0.934
linear_regression,2.298,10.164,0.933
ridge,2.298,10.164,0.933
lasso,2.342,10.394,0.929


## 4. Table 4 — top-k zone-hour overlap (all four models + random baseline)

For k = 50, 200, 1,000, and 5,000: what percentage of the true top-k busiest zone-hours
each model's own ranking recovers, against an unranked/random baseline of the same list
size.

In [5]:
n_total = len(eval_df)
rows = []
for k in [50, 200, 1000, 5000]:
    row = {"k": k, "random": 100 * k / n_total}
    for name, col in pred_cols.items():
        row[name] = 100 * top_k_zone_hour_overlap(eval_df, k, "actual", col)
    rows.append(row)
table4 = pd.DataFrame(rows).set_index("k")
print("Table 4 -- top-k overlap (%), all models vs random baseline")
table4.round(3)

Table 4 -- top-k overlap (%), all models vs random baseline


,random,gradient_boosting,linear_regression,ridge,lasso
k,,,,,
50,0.003,40.0,38.00,38.00,40.00
200,0.011,42.0,55.00,55.00,56.00
1000,0.053,50.3,60.00,60.00,59.20
5000,0.265,72.5,75.42,75.42,75.54


## 5. Table 5 — feature importance (GB) / retained-eliminated status (LASSO)

### Gradient Boosting permutation importance

Computed on a 20,000-row subsample of the test set (permutation importance re-predicts
the full sample once per feature per repeat — computationally prohibitive at the full
1.89M-row scale).

In [6]:
gb_model = models["gradient_boosting"]
X_test_gb, y_test_gb = build_feature_matrix(test_df, "gradient_boosting")

rng = np.random.default_rng(42)
sample_idx = rng.choice(len(X_test_gb), size=20_000, replace=False)

result = permutation_importance(
    gb_model, X_test_gb.iloc[sample_idx], y_test_gb.iloc[sample_idx],
    n_repeats=3, random_state=42, n_jobs=-1,
)
gb_importances = pd.Series(result.importances_mean, index=X_test_gb.columns).sort_values(ascending=False)
print("Gradient Boosting permutation importance (full ranking):")
print(gb_importances.to_string())

baseline_imp = gb_importances["log1p_baseline_pickup_count"]
next_largest = gb_importances.drop("log1p_baseline_pickup_count").max()
print(f"\nBaseline importance: {baseline_imp:.4f}, next largest: {next_largest:.6f}, ratio: {baseline_imp / next_largest:.0f}x")

Gradient Boosting permutation importance (full ranking):
log1p_baseline_pickup_count    1.816545
hour_of_day                    0.004628
day_of_week                    0.001704
is_holiday                     0.001633
month                          0.001426
pu_location_id                 0.000783
taxi_type                      0.000316
relh                           0.000253
dwpf                           0.000246
tmpf                           0.000134
p01i                           0.000114
gust                           0.000062
event_count                    0.000052
weather_severity               0.000026
sknt                           0.000005
vsby                           0.000004
event_present                  0.000000

Baseline importance: 1.8165, next largest: 0.004628, ratio: 392x


### LASSO retained/eliminated status, by feature group

In [7]:
lasso_coefs = pd.Series(lasso_model.coef_, index=X_train_scaled.columns)

FEATURE_GROUPS = {
    "Baseline": ["log1p_baseline_pickup_count"],
    "Hour of day": [c for c in lasso_coefs.index if c.startswith("hour_of_day_")],
    "Holiday flag": ["is_holiday"],
    "Day of week": [c for c in lasso_coefs.index if c.startswith("day_of_week_")],
    "Month": [c for c in lasso_coefs.index if c.startswith("month_")],
    "Zone/Borough": [c for c in lasso_coefs.index if c.startswith("borough_")],
    "Taxi type": [c for c in lasso_coefs.index if c.startswith("taxi_type_")],
    "Weather severity": [c for c in lasso_coefs.index if c.startswith("weather_severity_")],
    "Weather (numeric)": ["tmpf", "dwpf", "relh", "sknt", "gust", "vsby", "p01i"],
    "Events": ["event_count", "event_present"],
}

print(f"{'Group':<20}{'Status':<14}{'Retained/Total':<16}Retained columns")
for group, cols in FEATURE_GROUPS.items():
    nonzero = [c for c in cols if lasso_coefs.get(c, 0) != 0]
    status = "Retained" if nonzero else "Eliminated"
    print(f"{group:<20}{status:<14}{len(nonzero)}/{len(cols):<14}{nonzero}")

print(f"\nBaseline coefficient: {lasso_coefs['log1p_baseline_pickup_count']:.3f}")
print(f"Non-zero coefficients overall: {(lasso_coefs != 0).sum()} / {len(lasso_coefs)}")

Group               Status        Retained/Total  Retained columns
Baseline            Retained      1/1             ['log1p_baseline_pickup_count']
Hour of day         Eliminated    0/24            []
Holiday flag        Retained      1/1             ['is_holiday']
Day of week         Eliminated    0/7             []
Month               Retained      7/12            ['month_1', 'month_2', 'month_3', 'month_5', 'month_7', 'month_8', 'month_9']
Zone/Borough        Retained      4/6             ['borough_Brooklyn', 'borough_Manhattan', 'borough_Queens', 'borough_Staten Island']
Taxi type           Retained      2/2             ['taxi_type_yellow', 'taxi_type_green']
Weather severity    Eliminated    0/3             []
Weather (numeric)   Retained      2/7             ['sknt', 'vsby']
Events              Eliminated    0/2             []

Baseline coefficient: 1.213
Non-zero coefficients overall: 17 / 65


## 6. Section 5.3 — DTDI prediction error, all four models + Table (reliability under severe weather)

DTDI (demand / baseline) prediction error, computed for all four models: how compressed
each model's predicted DTDI is relative to actual DTDI, how each model's accuracy
changes under severe vs. normal weather, and how much of the true severe-weather DTDI
drop each model's own predictions actually reproduce. This is the evidence behind Table
\ref{tab:reliability} in the report.

In [8]:
from src.evaluate import regression_metrics_by_group

reliable = eval_df[eval_df["baseline_pickup_count"] >= MIN_BASELINE_FOR_DTDI].copy()
reliable["dtdi_actual"] = compute_dtdi(reliable["actual"], reliable["baseline_pickup_count"])
actual_std = reliable["dtdi_actual"].std()

print(f"Reliable rows (baseline >= {MIN_BASELINE_FOR_DTDI}): {len(reliable):,} of {len(eval_df):,}")
print(f"Actual DTDI std dev: {actual_std:.4f}\n")

print("MAE/RMSE/R2 by weather severity, per model")
mae_ratios = {}
for name, col in pred_cols.items():
    metrics = regression_metrics_by_group(eval_df, "weather_severity", "actual", col)
    mae_ratios[name] = metrics.loc["severe", "MAE"] / metrics.loc["normal", "MAE"]
    print(f"\n{name}:")
    print(metrics.round(3))

print("\nMAE ratio (severe/normal), per model:")
for name, ratio in mae_ratios.items():
    print(f"  {name}: {ratio:.2f}x")

Reliable rows (baseline >= 1.0): 576,675 of 1,888,340
Actual DTDI std dev: 0.7619

MAE/RMSE/R2 by weather severity, per model

gradient_boosting:
                    MAE    RMSE     R2        n
weather_severity                               
mild              2.306  10.627  0.915   322438
normal            2.120   9.578  0.942  1493840
severe            3.393  16.105  0.802    72062



linear_regression:
                    MAE    RMSE     R2        n
weather_severity                               
mild              2.395  10.829  0.912   322438
normal            2.216   9.615  0.942  1493840
severe            3.567  16.311  0.797    72062

ridge:
                    MAE    RMSE     R2        n
weather_severity                               
mild              2.395  10.829  0.912   322438
normal            2.216   9.615  0.942  1493840
severe            3.567  16.311  0.797    72062



lasso:
                    MAE    RMSE     R2        n
weather_severity                               
mild              2.429  11.018  0.909   322438
normal            2.263   9.864  0.939  1493840
severe            3.606  16.452  0.794    72062

MAE ratio (severe/normal), per model:
  gradient_boosting: 1.60x
  linear_regression: 1.61x
  ridge: 1.61x
  lasso: 1.59x


In [9]:
dtdi_by_severity = reliable.groupby("weather_severity", observed=True)["dtdi_actual"].agg(
    ["mean", "median", "count"]
)
print("Actual DTDI by weather severity:")
print(dtdi_by_severity)

normal_dtdi = dtdi_by_severity.loc["normal", "mean"]
severe_dtdi = dtdi_by_severity.loc["severe", "mean"]
true_drop = normal_dtdi - severe_dtdi
print(f"\nMean actual DTDI: normal={normal_dtdi:.3f}, severe={severe_dtdi:.3f} (drop of {true_drop:.3f})")

print("\nPredicted DTDI by weather severity, per model, compression ratio, and drop captured:")
for name, col in pred_cols.items():
    reliable[f"dtdi_pred_{name}"] = compute_dtdi(reliable[col], reliable["baseline_pickup_count"])
    pred_by_severity = reliable.groupby("weather_severity", observed=True)[f"dtdi_pred_{name}"].mean()
    pred_drop = pred_by_severity["normal"] - pred_by_severity["severe"]
    captured_pct = 100 * pred_drop / true_drop
    pred_std = reliable[f"dtdi_pred_{name}"].std()
    std_ratio = actual_std / pred_std
    print(
        f"  {name}: predicted DTDI normal={pred_by_severity['normal']:.3f}, "
        f"severe={pred_by_severity['severe']:.3f} (drop of {pred_drop:.3f}, "
        f"~{captured_pct:.0f}% of true drop) | std ratio (actual/predicted) = {std_ratio:.2f}x"
    )

Actual DTDI by weather severity:
                      mean    median   count
weather_severity                            
mild              0.993937  0.891720   98217
normal            1.000368  0.944099  456428
severe            0.897516  0.846922   22030

Mean actual DTDI: normal=1.000, severe=0.898 (drop of 0.103)

Predicted DTDI by weather severity, per model, compression ratio, and drop captured:
  gradient_boosting: predicted DTDI normal=0.920, severe=0.906 (drop of 0.015, ~14% of true drop) | std ratio (actual/predicted) = 4.89x
  linear_regression: predicted DTDI normal=0.939, severe=0.941 (drop of -0.002, ~-2% of true drop) | std ratio (actual/predicted) = 20.41x
  ridge: predicted DTDI normal=0.939, severe=0.941 (drop of -0.002, ~-2% of true drop) | std ratio (actual/predicted) = 20.41x
  lasso: predicted DTDI normal=0.934, severe=0.939 (drop of -0.005, ~-5% of true drop) | std ratio (actual/predicted) = 28.25x


## Notes

- LASSO's tuned alpha is derived fresh here via `tune_lasso_alpha`, not hardcoded from an
  earlier session's result (the old notebook asserted `alpha=0.001` directly).
- Table 3/4's four-model comparison and Table 5's two perspectives (GB permutation
  importance vs. LASSO retained/eliminated) are both computed from the exact same
  `eval_df` / `lasso_model` built above — no cross-notebook stitching required.
- Table \ref{tab:reliability} (Section 5.3) reuses this same `eval_df`, extended with
  `weather_severity` groupings and DTDI compression ratios for all four models. It was
  previously a two-panel figure (DTDI error histogram + spatial map, Gradient Boosting
  only) — replaced because a table conveys the same compression finding more compactly,
  and extending the analysis to all four models plus the severe-weather comparison did
  not fit alongside a figure within the report's page budget. The GB-only figure code is
  no longer here; the underlying `dtdi_error`/spatial-map computation was cut, not the
  finding itself, which the table still reports (GB's ~4.89x compression ratio, matching
  the old figure's "roughly 5x" claim).
- Recommendation 1's numbers (Section 6.1: top-50/top-1000 overlap, the 25 April 2026
  top-20 list) reuse this notebook's `eval_df` and are computed in notebook `08`.
  Recommendation 2's numbers (Section 6.2: severe-weather MAE/DTDI evidence) are the same
  underlying finding as Table \ref{tab:reliability} above, but computed independently in
  notebook `08` (its own model loading and LASSO refit) so that notebook stays
  self-contained.
- **Gradient Boosting's exact numbers vary slightly run-to-run** (confirmed by running
  this notebook twice from a clean state): MAE/RMSE/R2 in Table 3, the GB column in
  Table 4, GB's permutation importances in Table 5, and the DTDI statistics in Section 6
  (which use GB's predictions) all show small deltas (e.g. MAE within +/-0.02, top-k
  overlap within a few percentage points) between runs, even with a fixed
  `random_state`. Every other model, and the entire data pipeline upstream of modelling,
  reproduces bit-for-bit identically across runs. This is a known characteristic of
  sklearn's `HistGradientBoostingRegressor` under multi-threading, not a bug in this
  notebook or pipeline -- if your own re-run's GB numbers differ slightly from what's
  shown here, that's expected.